In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from pydantic import BaseModel ,Field


In [ ]:
load_dotenv()
Groq_KEY = os.getenv("Groq_KEY")


In [ ]:
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=Groq_KEY
)

In [ ]:
class Reviewstate(TypedDict):
    review:str
    sentiment:Literal['positive','negative']
    diagnosis:dict
    response:str
    

In [ ]:
class sentimentSchema(BaseModel):
    sentiment:Literal['positive','negative']=Field(description="Sentiment of the review")
    
    

In [ ]:
structuered_model=model.with_structured_output(sentimentSchema)

In [ ]:
prompt="""what is the sentiment of the review? this is too good to be true. I am very happy with the product.
I will recommend it to my friends and family."""


In [ ]:
structuered_model.invoke(prompt).sentiment

In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(
        description="The category of issue mentioned in the review"
    )
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(
        description="The emotional tone expressed by the user"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent or critical the issue appears to be"
    )

In [ ]:
structured_diagnosis_model = model.with_structured_output(DiagnosisSchema)


In [ ]:
prompt2="""Analyze the following review and provide a structured diagnosis of the issue mentioned, including the issue type, tone, and urgency.
Review: "The app crashes every time I try to upload a photo. This is extremely frustrating and I need a fix ASAP!" """

In [ ]:
structured_diagnosis_model.invoke(prompt2)

In [ ]:
def find_sentiment(state: Reviewstate):
    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_diagnosis_model.invoke(prompt).sentiment
    return {'sentiment': sentiment}


In [ ]:
def positive_response(state: Reviewstate):
    prompt = f"""Write a warm thank-you message in response to this review:
\n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""
    response = model.invoke(prompt).content
    return {'response': response}

In [ ]:
def run_diagnosis(state: Reviewstate):
    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_diagnosis_model.invoke(prompt)
    return {'diagnosis': response.model_dump()}

In [ ]:
def negative_response(state: Reviewstate):
    diagnosis = state['diagnosis']
    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content
    return {'response': response}

In [ ]:
def check_sentiment(state: Reviewstate) -> Literal["positive_response", "run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [ ]:
graph=StateGraph(Reviewstate, START, END)
#add nodes

graph.add_node(find_sentiment, "Find Sentiment")
graph.add_node(run_diagnosis, "Run Diagnosis")
graph.add_node(positive_response, "Positive Response")
graph.add_node(negative_response, "Negative Response")


#add edges
graph.add_edge(START, "Find Sentiment")
graph.add_edge("Find Sentiment", check_sentiment)

graph.add_edge("Positive Response", END)
graph.add_edge("Run Diagnosis", "Negative Response")
graph.add_edge("Negative Response", END)

In [ ]:
workflow=graph.compile()

In [ ]:
workflow